# Objectives
Create new features that capture potentially useful patterns.

Encode categorical variables so models can use them.

Split the data into training and testing sets before any scaling.

Scale numerical features when the chosen model benefits from it.

In [2]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
import os

print(os.getcwd())

/var/www/filebrowser/.projects/7dc8a4f6-a12e-46aa-a43e-0802e3ae41b0


In [4]:
os.listdir()

['.ipynb_checkpoints',
 'Untitled1.ipynb',
 'clean_credit_risk.csv',
 '01_Data_Exploration1.ipynb',
 'Untitled.ipynb',
 '03_Feature_Engineering.ipynb',
 '02_EDA.ipynb',
 '01_Data_Cleaning.ipynb',
 'credit_risk_dataset.csv']

In [4]:
df = pd.read_csv("clean_credit_risk.csv")

In [6]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


In [7]:
threshold = df["loan_percent_income"].quantile(0.75)

df["high_loan_income_ratio"] = (
    df["loan_percent_income"] > threshold
).astype(int)

In [8]:
df["long_employment"] = (
    df["person_emp_length"] >= 5
).astype(int)

In [9]:
df["young_borrower"] = (
    df["person_age"] < 25
).astype(int)

In [10]:
interest_threshold = df["loan_int_rate"].quantile(0.75)

df["high_interest_rate"] = (
    df["loan_int_rate"] > interest_threshold
).astype(int)

In [11]:
df["income_band"] = pd.qcut(
    df["person_income"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

In [12]:
new_features = [
    "high_loan_income_ratio",
    "long_employment",
    "young_borrower",
    "high_interest_rate",
    "income_band"
]

df[new_features].head()

,high_loan_income_ratio,long_employment,young_borrower,high_interest_rate,income_band
0,0,1,1,0,Low
1,1,0,0,0,Low
2,1,0,1,1,High
3,1,1,1,1,Medium
4,1,0,1,0,Low


In [13]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

# income_band is categorical even though created above
categorical_columns.append("income_band")

categorical_columns = list(set(categorical_columns))

print(categorical_columns)

['cb_person_default_on_file', 'person_home_ownership', 'loan_grade', 'income_band', 'loan_intent']


In [14]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

In [15]:
X = df_encoded.drop("loan_status", axis=1)
y = df_encoded["loan_status"]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

Training set: (25217, 29)
Testing set : (6305, 29)


In [18]:
numerical_columns = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

In [19]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test_scaled[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [20]:
print("Original Training Shape :", X_train.shape)
print("Scaled Training Shape   :", X_train_scaled.shape)

print("\nTarget Distribution")
print(y_train.value_counts(normalize=True))

Original Training Shape : (25217, 29)
Scaled Training Shape   : (25217, 29)

Target Distribution
0    0.784074
1    0.215926
Name: loan_status, dtype: float64


In [5]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

df_encoded.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
0,21,9600,5.0,1000,11.14,0,0.10,2,0,1,...,0,0,0,1,0,0,0,0,0,0
1,25,9600,1.0,5500,12.87,1,0.57,3,0,0,...,1,0,0,0,1,0,0,0,0,0
2,23,65500,4.0,35000,15.23,1,0.53,2,0,0,...,1,0,0,0,1,0,0,0,0,0
3,24,54400,8.0,35000,14.27,1,0.55,4,0,0,...,1,0,0,0,1,0,0,0,0,1
4,21,9900,2.0,2500,7.14,1,0.25,2,0,1,...,0,0,1,0,0,0,0,0,0,0


In [6]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31522 entries, 0 to 31521
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   person_age                   31522 non-null  int64  
 1   person_income                31522 non-null  int64  
 2   person_emp_length            31522 non-null  float64
 3   loan_amnt                    31522 non-null  int64  
 4   loan_int_rate                31522 non-null  float64
 5   loan_status                  31522 non-null  int64  
 6   loan_percent_income          31522 non-null  float64
 7   cb_person_cred_hist_length   31522 non-null  int64  
 8   person_home_ownership_OTHER  31522 non-null  uint8  
 9   person_home_ownership_OWN    31522 non-null  uint8  
 10  person_home_ownership_RENT   31522 non-null  uint8  
 11  loan_intent_EDUCATION        31522 non-null  uint8  
 12  loan_intent_HOMEIMPROVEMENT  31522 non-null  uint8  
 13  loan_intent_MEDI

In [7]:
df_encoded.to_csv("credit_processed.csv", index=False)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
